# Homework: Muon and Newton–Schulz

В этом задании нужно реализовать ортогонализацию Ньютона–Шульца (Newton–Schulz) и оптимизатор **Muon** — метод для скрытых слоёв нейросетей, в котором направление обновления строится через ортогонализацию момента градиентов.

**Идея Muon:** вместо шага по градиенту (или по моменту, как в SGD с моментом) строится момент $B_t = \mu B_{t-1} + g_t$, к нему применяется **k итераций Newton–Schulz**, дающие ортогональную (полярную) компоненту $O_t$; веса обновляются как $\theta_t \leftarrow \theta_t - \gamma O_t$. Так сохраняется ортогональность весов и стабильность сингулярных чисел при обучении. Обычно Muon применяют только к 2D-параметрам (матрицам весов); bias и др. оптимизируют, например, AdamW.

**Ссылки:**
- [torch.optim.Muon](https://docs.pytorch.org/docs/stable/generated/torch.optim.Muon.html) — эталонная реализация и описание алгоритма в PyTorch 2.10.
- Jordan et al. 2024 — Muon: An optimizer for hidden layers in neural networks.
- Muon is Scalable for LLM Training (Moonshot); OpenReview ICLR 2026 — Convergence of Muon with Newton–Schulz.


In [ ]:
%matplotlib inline
import math
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
# torch.linalg.svd for optional comparisons


## 📐 Свойства ортогональных матриц

**Определение.** Матрица $Q \in \mathbb{R}^{m \times n}$ ортогональна (ортонормированные столбцы/строки), если $Q^T Q = I_n$ при $m \ge n$ или $Q Q^T = I_m$ при $m \le n$.

**Следствия:** ✅
- $\|Q v\| = \|v\|$ — сохранение нормы
- сингулярные числа = 1 (или 0)
- $\kappa(Q) = 1$ при полном ранге

То есть ортогональное преобразование не сжимает и не растягивает направления — градиенты не взрываются и не затухают.

> PS В классической терминологии прямоугольные матрицы с ортонормированными столбцами/строками называют полуортогональными. А ортогональными называют только квадратные матрицы.

In [13]:
import torch
import math

# Примеры ортогональных и неортогональных матриц

# Ортогональные матрицы: Q^T Q = I (для m >= n) или Q Q^T = I (для m <= n).
# Сингулярные числа ортогональной матрицы полного ранга равны 1.

def print_svdvals(m, name):
    print(f"Singular values of {name}:", [f"{s:.3f}" for s in torch.linalg.svdvals(m).tolist()])

print("Единичная матрица — ортогональна")
I = torch.eye(3)
print("I @ I.T:", (I @ I.T - I).norm().item())
print("")

print("Случайная ортогональная матрица через QR-разложение")
Q, _ = torch.linalg.qr(torch.randn(4, 4))
print("Q orthogonality (Q.T @ Q - I) fro:", (Q.T @ Q - torch.eye(4)).norm(p="fro").item())
print_svdvals(Q, "Q")
print("")

print("Матрица поворота в плоскости (ортогональна, det = 1)")
angle = math.pi / 4
R = torch.tensor([[math.cos(angle), -math.sin(angle)], [math.sin(angle), math.cos(angle)]])
print("R orthogonality:", (R.T @ R - torch.eye(2)).norm().item())
print_svdvals(R, "R")
print("")

print("-----\n\nНеортогональные матрицы: Q^T Q != I, сингулярные числа не равны 1.\n")

print("Случайная матрица — в общем случае не ортогональна")
A = torch.randn(3, 3) * 0.5
print("A orthogonality (A.T @ A - I) fro:", (A.T @ A - torch.eye(3)).norm(p="fro").item())
print_svdvals(A, "A")
print("")

print("Диагональная матрица с разными числами — не ортогональна (если не ±1 на диагонали)")
D = torch.diag(torch.tensor([1.0, 2.0, 0.5]))
print("D orthogonality:", (D.T @ D - torch.eye(3)).norm().item())
print_svdvals(D, "D")


Единичная матрица — ортогональна
I @ I.T: 0.0

Случайная ортогональная матрица через QR-разложение
Q orthogonality (Q.T @ Q - I) fro: 1.9504824422256206e-07
Singular values of Q: ['1.000', '1.000', '1.000', '1.000']

Матрица поворота в плоскости (ортогональна, det = 1)
R orthogonality: 8.618236790880474e-08
Singular values of R: ['1.000', '1.000']

-----

Неортогональные матрицы: Q^T Q != I, сингулярные числа не равны 1.

Случайная матрица — в общем случае не ортогональна
A orthogonality (A.T @ A - I) fro: 1.9502880573272705
Singular values of A: ['1.562', '0.382', '0.021']

Диагональная матрица с разными числами — не ортогональна (если не ±1 на диагонали)
D orthogonality: 3.092329263687134
Singular values of D: ['2.000', '1.000', '0.500']


## Норма Фробениуса

**Определение.** Для матрицы $A \in \mathbb{R}^{m \times n}$ норма Фробениуса
$$
\|A\|_F = \sqrt{\sum_{i,j} A_{ij}^2} = \sqrt{\operatorname{tr}(A^T A)} = \sqrt{\sum_k \sigma_k^2},
$$
где $\sigma_k$ — сингулярные числа $A$.

**Свойства:**
- $\|A\|_F^2$ — сумма квадратов всех элементов матрицы
- Инвариантность к ортогональным преобразованиям: $\|U A V^T\|_F = \|A\|_F$ для ортогональных $U$, $V$
- Связь со спектральной нормой: $\|A\|_{\mathrm{op}} \le \|A\|_F \le \sqrt{\min(m,n)}\,\|A\|_{\mathrm{op}}$

В итерации Ньютона–Шульца норму Фробениуса используют для масштабирования матрицы (чтобы сингулярные числа попали в зону быстрой сходимости) и для проверки критериев остановки.


In [ ]:
### Пример вычисления pytorch
import torch
X = torch.rand(64,64)
X.norm(p="fro").item()

36.7525749206543

## Полином Ньютона–Шульца

<details>
  <summary>Полином Ньютона–Шульца</summary>

Полярный фактор можно получить через SVD: если $X = U S V^T$, то $\operatorname{Polar}(X) = U V^T$. Эквивалентно, $\operatorname{Polar}(X) = (X X^T)^{-1/2} X$ (при полном ранге): матрица $(X X^T)^{-1/2}$ действует как «нормализация» сингулярных чисел к 1. Вычислять $(X X^T)^{-1/2}$ напрямую дорого; вместо этого используют итерацию, в которой на каждом шаге применяется **полином**, приближающий функцию $\lambda \mapsto 1/\sqrt{\lambda}$ в окрестности $\lambda = 1$. Где $\lambda$ - это собственное число матрицы $ X X^T $

**Полином Ньютона–Шульца степени $\kappa$** — это $p_\kappa(\lambda) = \sum_{s=0}^{\kappa} c_s (1-\lambda)^s$ с коэффициентами
$$
c_s = \frac{(2s)!}{4^s (s!)^2}.
$$
Он совпадает с частичной суммой ряда Тейлора функции $1/\sqrt{\lambda}$ в точке $\lambda = 1$ (разложение по $(1-\lambda)$). Чем больше $\kappa$, тем лучше приближение на отрезке около 1.

**Итерация.** После масштабирования $X_0 = X / \max(1, \|X\|_F)$ сингулярные числа $X_0$ лежат в $(0, 1]$, поэтому собственные числа $X_j X_j^T$ близки к 1. Шаг Ньютона–Шульца:
$$
X_{j+1} = p_\kappa(X_j X_j^T)\, X_j.
$$
Здесь $p_\kappa$ применяется к симметричной положительно определённой матрице $X_j X_j^T$ (через собственное разложение или полиномиальные комбинации матрицы). Последовательность $X_j$ сходится к полярному фактору $X_0$, т.е. к ортогональной матрице с тем же действием на подпространство образа $X$.

</details>


## Задание 1 — Ортогонализация Ньютона–Шульца (100 points)


**Цель:** по матрице $X$ вычислить **полярный фактор** (ортогональную матрицу $U V^T$ из SVD $X = U S V^T$) без полного SVD — итерационно, через метод Ньютона–Шульца.


**Почему полярный фактор равен $(X X^T)^{-1/2} X$**

<details>
  <summary>Почему?</summary>

По SVD: $X = U S V^T$, значит $X X^T = U S^2 U^T$ (собственные значения $X X^T$ — это $\sigma_k^2$). Степень матрицы задаётся через собственные значения: $(X X^T)^{-1/2} = U S^{-1} U^T$. Тогда
$$
(X X^T)^{-1/2} X = U S^{-1} U^T \cdot U S V^T = U V^T = \operatorname{Polar}(X).
$$
Степень $-1/2$ нужна именно чтобы «сократить» $S$ в $X$ и получить только ортогональную часть $U V^T$. Считать $(X X^T)^{-1/2}$ через полное собственное разложение дорого — вместо этого приближаем функцию $\lambda \mapsto 1/\sqrt{\lambda}$ полиномом и применяем её к $X_j X_j^T$ в итерации.

</details>

**Что такое $\lambda$ и зачем полином $1/\sqrt{\lambda}$**

<details>
  <summary>Что и зачем?</summary>

$\lambda$ — это **собственное значение** матрицы $X X^T$ (т.е. $\sigma^2$ для сингулярного числа $\sigma$ матрицы $X$). Чтобы получить $(X X^T)^{-1/2}$, нужно каждое такое $\lambda$ превратить в $\lambda^{-1/2} = 1/\sqrt{\lambda}$. Полином Ньютона–Шульца $p_\kappa(\lambda)$ — это **обрезанный ряд Тейлора** функции $1/\sqrt{\lambda}$ в точке $\lambda = 1$ (по переменной $u = 1 - \lambda$): разложение $(1-u)^{-1/2} = \sum_{s\ge 0} c_s (1-\lambda)^s$ с $c_s = (2s)!/(4^s (s!)^2)$. В окрестности $\lambda = 1$ частичная сумма хорошо приближает $1/\sqrt{\lambda}$; поэтому перед итерацией делают масштабирование по норме Фробениуса — чтобы все $\lambda$ оказались недалеко от 1.

**Полиномиальная форма vs коэффициенты $(a,b,c)$ в PyTorch:** это две записи одной и той же итерации. В PyTorch используют параметры $(a,b,c)$ и $\varepsilon$ для численной стабильности. В этом задании достаточно реализовать через указанную полиномиальную форму $p_\kappa(\lambda) = \sum_{s=0}^{\kappa} c_s (1-\lambda)^s$.

</details>



**Алгоритм** (см. [torch.optim.Muon](https://docs.pytorch.org/docs/stable/generated/torch.optim.Muon.html), NS operator):
1. Масштабирование: $X_0 = X / \max(1, \|X\|_F)$ (чтобы сингулярные числа были в зоне сходимости).
2. **Итерация:** $X_{j+1} = p_\kappa(X_j X_j^T)\, X_j$, где $p_\kappa(\lambda) = \sum_{s=0}^{\kappa} c_s (1-\lambda)^s$, $c_s = (2s)! / (4^s (s!)^2)$. Шаг повторяют `num_steps` раз.

**Сигнатура:** реализуйте функцию `schulz_orthogonalize(X, num_steps, kappa)` (вход: 2D тензор, выход: тензор той же формы; для «высоких» и «широких» матриц результат имеет ортогональные столбцы или строки соответственно).


In [ ]:
import math
import torch


def schulz_orthogonalize(X, num_steps, kappa):
    """
    Newton–Schulz orthogonalization: приближение полярного фактора матрицы X.

    Алгоритм (см. torch.optim.Muon, NS operator):
    - Масштабирование: X_0 = X / max(1, ||X||_F).
    - num_steps раз: X_{j+1} = p_kappa(X_j X_j^T) X_j, где p_kappa — полином
      Ньютона–Шульца степени kappa, приближающий 1/sqrt(λ) около 1:
      p_kappa(λ) = sum_{s=0}^{kappa} c_s (1-λ)^s, c_s = (2s)! / (4^s (s!)^2).

    Вход: X — 2D тензор (m×n). Выход: тензор той же формы (ортогональные
    строки для wide, ортогональные столбцы для tall).

    Ссылки: https://docs.pytorch.org/docs/stable/generated/torch.optim.Muon.html,
    OpenReview ICLR 2026 "Convergence of Muon with Newton–Schulz".
    """
    assert X.dim() == 2, "X must be 2D"
    # TODO: реализовать итерацию Newton–Schulz (см. описание выше и ячейку с теорией)
    raise NotImplementedError("Implement Newton–Schulz orthogonalization")



In [ ]:
X = torch.randn(64, 256)

# Протестируем! После ортогонализации сингулярные числа должны быть близки к 1
print_svdvals(X, "X")

kappa = 2

# При увеличении кол-ва шагов видно, как сходится алгоритм
for ns_steps in range(1, 6):
    X_ns = schulz_orthogonalize(X, ns_steps, kappa)

    print_svdvals(X_ns, f"X_ns[steps={ns_steps}]")

In [ ]:
# Только для ноутбука: сделаем schulz_orthogonalize из ячейки выше доступным для import в следующей ячейке
import sys
import types
_mod = types.ModuleType("schulz_orthogonalize")
_mod.schulz_orthogonalize = schulz_orthogonalize
sys.modules["schulz_orthogonalize"] = _mod

## Задание 2 — Реализация Muon (100 points)

**Алгоритм (по [torch.optim.Muon](https://docs.pytorch.org/docs/stable/generated/torch.optim.Muon.html)):**
Вход: learning rate $\gamma$, (опционально) weight decay $\lambda$, momentum $\mu$, число шагов Newton–Schulz $k$, параметры $\theta$.

1. Инициализация: буфер момента $B_0 = 0$.
2. На шаге $t$:
   - $g_t = \nabla_\theta f_t(\theta_{t-1})$;
   - $B_t = \mu B_{t-1} + g_t$;
   - при Nesterov: $\tilde{B}_t = g_t + \mu B_t$, иначе $\tilde{B}_t = B_t$;
   - $O_t = \mathrm{NS}_k(\tilde{B}_t; \varepsilon)$ — $k$ итераций Newton–Schulz с числовой стабилизацией $\varepsilon$;
   - (опционально) decoupled weight decay: $\theta \leftarrow \theta - \gamma\lambda\theta$;
   - (опционально) корректировка $\gamma$ по форме матрицы (AdjustLR), чтобы RMS обновления был согласован для прямоугольных матриц;
   - $\theta_t \leftarrow \theta_t - \gamma O_t$.

**В этой работе:** достаточно реализовать базовый вариант: момент $M_t = \beta M_{t-1} + g_t$, затем $O_t = \texttt{schulz\_orthogonalize}(M_t, \texttt{num\_steps}, \texttt{kappa})$, обновление $W_t = W_{t-1} - \eta O_t$. Применять **только к 2D-параметрам** (матрицам весов); параметры с `ndim != 2` (bias, embedding и т.д.) не обновлять. На практике остальные веса оптимизируются другим оптимизатором.


In [ ]:
import torch
from schulz_orthogonalize import schulz_orthogonalize


class Muon(torch.optim.Optimizer):
    """
    Muon: оптимизатор с ортогонализацией момента (Newton–Schulz) для 2D-параметров.

    Алгоритм (упрощённый вариант torch.optim.Muon):
    - Момент: M_t = beta * M_{t-1} + g_t.
    - O_t = schulz_orthogonalize(M_t, num_steps, kappa).
    - W_t = W_{t-1} - lr * O_t.
    Обновлять только параметры с ndim == 2 (матрицы весов); 1D (bias и т.д.) пропускать.

    Полный алгоритм (lr adjustment, weight decay, Nesterov, ns_coefficients):
    https://docs.pytorch.org/docs/stable/generated/torch.optim.Muon.html
    """

    def __init__(self, params, lr=0.01, beta=0.9, num_steps=2, kappa=2):
        # nesterov=False - не поддерживаем этот параметр
        defaults = dict(lr=lr, beta=beta, num_steps=num_steps, kappa=kappa)
        super().__init__(params, defaults)

    @torch.no_grad()
    def step(self):
        # TODO: для каждой группы параметров и каждого 2D-параметра p:
        # обновить state["momentum"], применить schulz_orthogonalize к моменту,
        # сделать p -= lr * orthogonalized. 1D-параметры пропускать.
        raise NotImplementedError("Implement Muon step")



## Задание 3 — Сравнение с Adam и сингулярные числа (дополнительное, не оценивается)

Обучите одну и ту же MLP с Adam и с Muon. На части шагов сохраняйте матрицы апдейтов и постройте графики сингулярных чисел (Adam vs Muon). При желании — сингулярные числа весов до обучения и кривые loss.


In [ ]:
# Optional: training loop and singular value plots

import matplotlib.pyplot as plt

def make_model():
    return nn.Sequential(
        nn.Linear(64, 32, bias=False),
        nn.ReLU(),
        nn.Linear(32, 10, bias=False),
    )

# opt_muon = Muon(model.parameters(), lr=0.01, beta=0.9, num_steps=5, kappa=3)
# opt_torch_muon = torcn.optim.Muon(model.parameters(), lr=0.01, beta=0.9, num_steps=5)
# opt_adam = torch.optim.Adam(model.parameters(), lr=0.01)
# ... collect update matrices, compute and compare SVD for updates and weights, plt.plot(singular_values)


## Запуск тестов в ноутбуке

Ниже те же проверки, что в checkhw (tests/muon/). Запустите ячейку после выполнения заданий 1 и 2. Используются `schulz_orthogonalize` и `Muon` из ячеек выше.

In [ ]:
# Same tests as in checkhw/tests/muon/ — run after cells above

import torch.nn as nn

def run_schulz_tests():
    """Tests for schulz_orthogonalize (from cell above)."""
    # test_schulz_signature_and_shape
    for m, n in [(32, 32), (64, 32), (16, 64)]:
        X = torch.randn(m, n) * 0.5
        Y = schulz_orthogonalize(X, num_steps=3, kappa=2)
        assert Y.shape == X.shape, f"shape mismatch: {Y.shape} vs {X.shape}"
        assert Y.device == X.device
        assert Y.dtype == X.dtype
    print("  schulz_signature_and_shape OK")
    # test_schulz_orthogonality_square
    torch.manual_seed(42)
    X = torch.randn(32, 32) * 0.5
    Y = schulz_orthogonalize(X, num_steps=5, kappa=2)
    I = torch.eye(32, device=Y.device, dtype=Y.dtype)
    diff = (Y @ Y.T - I).norm(p="fro").item()
    assert diff < 2.0, f"Y @ Y.T should be close to I, fro norm diff = {diff}"
    print("  schulz_orthogonality_square OK")
    # test_schulz_orthogonality_tall
    torch.manual_seed(43)
    X = torch.randn(48, 24) * 0.5
    Y = schulz_orthogonalize(X, num_steps=5, kappa=2)
    I = torch.eye(24, device=Y.device, dtype=Y.dtype)
    diff = (Y.T @ Y - I).norm(p="fro").item()
    assert diff < 1.0, f"Y.T @ Y should be close to I, fro norm diff = {diff}"
    print("  schulz_orthogonality_tall OK")
    # test_schulz_orthogonality_wide
    torch.manual_seed(44)
    X = torch.randn(24, 48) * 0.5
    Y = schulz_orthogonalize(X, num_steps=5, kappa=2)
    I = torch.eye(24, device=Y.device, dtype=Y.dtype)
    diff = (Y @ Y.T - I).norm(p="fro").item()
    assert diff < 1.0, f"Y @ Y.T should be close to I, fro norm diff = {diff}"
    print("  schulz_orthogonality_wide OK")
    # test_schulz_singular_values_polar_like
    torch.manual_seed(45)
    X = torch.randn(20, 20) * 0.4
    Y = schulz_orthogonalize(X, num_steps=6, kappa=2)
    S = torch.linalg.svdvals(Y)
    assert S.max() > 0.3, "result should have non-trivial singular values"
    for s in S:
        assert 0 <= s.item() <= 2.0, f"singular value {s.item()} out of range"
    print("  schulz_singular_values_polar_like OK")

def run_muon_tests():
    """Tests for Muon (from cell above)."""
    # test_muon_weights_stay_almost_orthogonal_after_10_steps
    torch.manual_seed(123)
    d = 32
    W = torch.randn(d, d)
    torch.nn.init.orthogonal_(W)
    W = nn.Parameter(W)
    opt = Muon([W], lr=0.01, beta=0.9, num_steps=3, kappa=2)
    for _ in range(10):
        opt.zero_grad()
        W.grad = torch.randn(d, d) * 0.1
        opt.step()
    S = torch.linalg.svdvals(W.detach())
    for s in S:
        assert s > 0.5 and s < 1.5, f"weight singular value {s.item()}"
    scale_sq = (W.detach() ** 2).sum() / d
    I = torch.eye(d, device=W.device, dtype=W.dtype)
    diff = (W.detach() @ W.detach().T - scale_sq * I).norm(p="fro").item()
    assert diff < 2.0 * (d ** 0.5), f"W W^T fro diff = {diff}"
    print("  muon_weights_stay_almost_orthogonal OK")
    # test_muon_step_changes_weights
    torch.manual_seed(124)
    linear = nn.Linear(16, 8, bias=False)
    w_before = linear.weight.data.clone()
    opt = Muon(linear.parameters(), lr=0.05, beta=0.9, num_steps=2, kappa=2)
    linear(torch.randn(2, 16)).sum().backward()
    opt.step()
    assert not torch.allclose(linear.weight.data, w_before), "weights should change"
    print("  muon_step_changes_weights OK")
    # test_muon_skips_1d_params
    torch.manual_seed(125)
    model = nn.Linear(8, 4, bias=True)
    bias_before = model.bias.data.clone()
    opt = Muon(model.parameters(), lr=0.01, beta=0.9, num_steps=2, kappa=2)
    model(torch.randn(2, 8)).sum().backward()
    opt.step()
    assert torch.allclose(model.bias.data, bias_before), "bias (1D) should be unchanged"
    print("  muon_skips_1d_params OK")
    # test_muon_close_to_torch_muon (skip if no torch.optim.Muon)
    if not hasattr(torch.optim, "Muon"):
        print("  muon_close_to_torch_muon SKIP (PyTorch < 2.10)")
        return
    torch.manual_seed(126)
    batch, steps = 4, 25
    data = [torch.randn(batch, 16) for _ in range(steps)]
    targets = [torch.randint(0, 4, (batch,)) for _ in range(steps)]
    def run_muon(OptimizerClass, **kwargs):
        model = nn.Linear(16, 4, bias=False)
        torch.manual_seed(126)
        with torch.no_grad():
            nn.init.orthogonal_(model.weight)
        opt = OptimizerClass(model.parameters(), **kwargs)
        for x, y in zip(data, targets):
            opt.zero_grad()
            loss = nn.functional.cross_entropy(model(x), y)
            loss.backward()
            opt.step()
        return model.weight.data.clone()
    w_student = run_muon(Muon, lr=0.02, beta=0.9, num_steps=2, kappa=2)
    torch.manual_seed(126)
    w_torch = run_muon(torch.optim.Muon, lr=0.02, weight_decay=0.0, momentum=0.9, nesterov=False, ns_steps=2)
    assert torch.allclose(w_student, w_torch, atol=0.1), "student Muon should be close to torch.optim.Muon"
    print("  muon_close_to_torch_muon OK")

print("Schulz tests:")
run_schulz_tests()
print("Muon tests:")
run_muon_tests()
print("All tests passed.")